In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# SHARED PROTEIN INTERACTION

# BioGrid Data
Downloaded from: https://downloads.thebiogrid.org/BioGRID/Release-Archive/BIOGRID-4.4.241/

In [2]:
def process_biogrid_ppi_data(data):
    # Filter interactions down to physical interactions between two human proteins
    hsapien = 9606
    biogrid_ppi = data[(data['Organism ID Interactor A']==hsapien) & (data['Organism ID Interactor B']==hsapien) &
                       (data['Experimental System Type']=='physical')]
    # Filter to columns of interest + rename
    biogrid_ppi = biogrid_ppi[['Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'Experimental System']]
    biogrid_ppi = biogrid_ppi.rename(columns={'Entrez Gene Interactor A':'A1_entrez', 'Entrez Gene Interactor B':'A2_entrez',
                                              'Experimental System':'experimental_system'})
    biogrid_ppi = biogrid_ppi.drop_duplicates().reset_index(drop=True).astype({'A1_entrez':'int', 'A2_entrez':'int'})
    
    # Filter out self interactions
    biogrid_ppi = biogrid_ppi[biogrid_ppi.A1_entrez != biogrid_ppi.A2_entrez].reset_index(drop=True)
    print('N interactions:', biogrid_ppi.shape[0])
    
    # Sort interacting genes - to find unique interaction pairs
    biogrid_unique = pd.DataFrame(np.sort(biogrid_ppi[['A1_entrez','A2_entrez']], axis=1), 
                                  columns=['A1_entrez', 'A2_entrez']).drop_duplicates()
    assert(biogrid_unique.shape[0] <= biogrid_ppi.shape[0])

    print('N interactions after sorting for unique pairs:', biogrid_unique.shape[0])
    print('N genes in interaction map:', pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).nunique())
    
    return biogrid_unique

In [3]:
biogrid_raw = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/BIOGRID-ALL-4.4.241.tab3.txt', sep='\t',  low_memory=False)
biogrid_raw[:1]
biogrid_unique = process_biogrid_ppi_data(biogrid_raw)
biogrid_genes = pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).unique()
biogrid_unique

N interactions: 993165
N interactions after sorting for unique pairs: 862452
N genes in interaction map: 19995


,A1_entrez,A2_entrez
0,2318,6416
1,88,84665
2,90,2339
3,2624,5371
4,6118,6774
...,...,...
993159,4085,26528
993160,4085,10521
993161,4085,10432
993162,9817,29843


In [9]:
## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df

In [10]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

biomarker_query = main_file_csv[['entrez_id_biomarker','entrez_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'entrez_id_biomarker': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
biomarker_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,biomarker_query)
biomarker_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_biomarker_query.csv',index=False)
biomarker_query_fet_shared_ppi

N. gene pairs that interact: 5056
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5728.0,5296.0,True,"{387, 6156, 2065, 4628, 3861, 8471, 408, 31, 4...",813,34,0.041820,14.036345
1,5728.0,3636.0,False,"{8451, 10376, 3850, 10381, 10383, 2065, 7316, ...",798,37,0.046366,17.368386
2,5728.0,7249.0,False,"{801, 5347, 867, 2885, 7879, 207, 2099, 5747, ...",770,14,0.018182,3.468631
3,5728.0,4771.0,False,"{4609, 5250, 6147, 8450, 8451, 8454, 6279, 628...",895,68,0.075978,35.100297
4,5728.0,63932.0,False,"{3329, 4609, 6160, 55709, 10528, 4000, 4001, 7...",689,19,0.027576,11.767982
...,...,...,...,...,...,...,...,...
144075,4893.0,440590.0,False,{},692,0,0.000000,0.000000
144076,4893.0,79699.0,False,"{1994, 440730}",763,2,0.002621,-0.112343
144077,4893.0,7791.0,False,"{5127, 11017, 9738, 10768, 440730, 57498, 1091...",960,29,0.030208,5.952641
144078,4893.0,23140.0,False,"{6117, 6118, 6119, 53834, 10618, 10551, 440730...",793,8,0.010088,1.097225


In [11]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target1_query = main_file_csv[['entrez_id_target1','entrez_id_query']]
target1_query = target1_query.rename(columns={
    'entrez_id_target1': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target1_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,target1_query)
target1_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target1_query.csv',index=False)
target1_query_fet_shared_ppi

N. gene pairs that interact: 2919
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5291.0,5296.0,True,"{84868, 3845, 6790, 7048, 2064, 2065, 8471, 10...",262,30,0.114504,38.971872
1,5291.0,3636.0,False,"{867, 1956, 2885, 5295, 2064, 2065, 4914, 5296...",271,9,0.033210,7.069189
2,5291.0,7249.0,False,"{867, 2885, 328, 5868, 8974, 207, 4914, 10870,...",220,9,0.040909,8.158703
3,5291.0,4771.0,False,"{1956, 3845, 6790, 7175, 328, 11113, 9928, 391...",396,12,0.030303,8.312495
4,5291.0,63932.0,False,{328},152,1,0.006579,0.594197
...,...,...,...,...,...,...,...,...
144075,5604.0,440590.0,False,{},262,0,0.000000,0.000000
144076,5604.0,79699.0,False,"{1994, 440730}",333,2,0.006006,0.511541
144077,5604.0,7791.0,False,"{55832, 440730, 7706, 1051, 10413, 699, 57664,...",534,25,0.046816,12.893010
144078,5604.0,23140.0,False,"{57664, 6117, 6790, 6118, 6119, 5071, 440730}",364,7,0.019231,3.010342


In [12]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target2_query = main_file_csv[['entrez_id_target2','entrez_id_query']]
target2_query = target2_query.rename(columns={
    'entrez_id_target2': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target2_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,target2_query)
target2_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target2_query.csv',index=False)
target2_query_fet_shared_ppi

N. gene pairs that interact: 913
N paralog pairs w/ 1+ interactor (A1 and/or A2): 141279
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,NaN,5296.0,False,{},217,0,0.000000,0.000000
1,NaN,3636.0,False,{},203,0,0.000000,0.000000
2,NaN,7249.0,False,{},152,0,0.000000,0.000000
3,NaN,4771.0,False,{},331,0,0.000000,0.000000
4,NaN,63932.0,False,{},76,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...
144075,5605.0,440590.0,False,{},277,0,0.000000,0.000000
144076,5605.0,79699.0,False,"{440730, 10987}",348,2,0.005747,0.477181
144077,5605.0,7791.0,False,"{440730, 7706, 23586, 10788, 55212, 9531, 2108...",554,20,0.036101,8.240295
144078,5605.0,23140.0,False,"{57664, 6117, 6118, 6119, 5071, 440730}",380,6,0.015789,2.195086


# STRING data

In [4]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]
string = string[string['combined_score'] >= 400].sort_values(by='combined_score', ascending=False)

string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)

# === HGNC lookup
gene_lookup = {}
def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)
    entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
    syms = []
    s = normalize_symbol(entry.get('symbol'))
    if s: syms.append(s)
    if pd.notnull(entry.get('prev_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
    if pd.notnull(entry.get('alias_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
    for sym in syms:
        if sym and sym not in gene_lookup:
            gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

string['hgnc_id_1'], string['ensembl_gene_id_1'], string['entrez_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'], string['entrez_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))

string


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_91821/3396990575.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2,hgnc_id_1,ensembl_gene_id_1,entrez_id_1,hgnc_id_2,ensembl_gene_id_2,entrez_id_2
0,ENSP00000259469,ENSP00000481646,0,0,0,959,989,0,514,999,RPL35,RPS10,HGNC:10344,ENSG00000136942,11224.0,HGNC:10383,ENSG00000124614,6204.0
1,ENSP00000229214,ENSP00000454836,0,0,0,759,994,360,243,999,KRR1,TBL3,HGNC:5176,ENSG00000111615,11103.0,HGNC:11587,ENSG00000183751,10607.0
2,ENSP00000260762,ENSP00000253861,0,0,0,60,978,800,999,999,EXOC6,EXOC4,HGNC:23196,ENSG00000138190,54536.0,HGNC:30389,ENSG00000131558,60412.0
3,ENSP00000294189,ENSP00000389103,0,0,0,991,994,0,502,999,RPL29,RPL23A,HGNC:10331,ENSG00000162244,6159.0,HGNC:10317,ENSG00000198242,6147.0
4,ENSP00000260762,ENSP00000370695,0,0,0,129,921,800,999,999,EXOC6,EXOC1,HGNC:23196,ENSG00000138190,54536.0,HGNC:30380,ENSG00000090989,55763.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1858939,ENSP00000311005,ENSP00000498019,0,0,0,0,400,0,0,400,MAPK7,RPS17,HGNC:6880,ENSG00000166484,5598.0,HGNC:10397,ENSG00000182774,6218.0
1858940,ENSP00000398495,ENSP00000248553,0,0,0,42,0,0,400,400,DIABLO-2,HSPB1,NaN,NaN,NA,HGNC:5246,ENSG00000106211,3315.0
1858941,ENSP00000357588,ENSP00000478070,0,0,0,400,0,0,0,400,CALHM5,CWC25,HGNC:21568,ENSG00000178033,254228.0,HGNC:25989,ENSG00000273559,54883.0
1858942,ENSP00000375622,ENSP00000395294,0,0,0,0,0,0,400,400,LAIR1,CD70,HGNC:6477,ENSG00000167613,3903.0,HGNC:11937,ENSG00000125726,970.0


In [16]:
string_unique = (
    string[["entrez_id_1", "entrez_id_2"]]
    .rename(columns={"entrez_id_1": "A1_entrez", "entrez_id_2": "A2_entrez"})
    .drop_duplicates()
    .reset_index(drop=True)
)

# Convert safely, making "NA" into NaN first
string_unique["A1_entrez"] = pd.to_numeric(string_unique["A1_entrez"], errors="coerce").astype("Int64")
string_unique["A2_entrez"] = pd.to_numeric(string_unique["A2_entrez"], errors="coerce").astype("Int64")

print('N interactions:', string_unique.shape[0])

# Drop rows with missing values before sorting
string_unique = string_unique.dropna(subset=["A1_entrez", "A2_entrez"])

# Sort interacting genes - to find unique interaction pairs
string_unique = (
    pd.DataFrame(np.sort(string_unique[['A1_entrez','A2_entrez']], axis=1), 
                 columns=['A1_entrez', 'A2_entrez'])
    .drop_duplicates()
)
string_unique = string_unique[string_unique["A1_entrez"] != string_unique["A2_entrez"]]

print('N interactions after sorting for unique pairs:', string_unique.shape[0])
print('N genes in interaction map:', pd.concat([string_unique.A1_entrez, string_unique.A2_entrez]).nunique())

string_unique


N interactions: 1840450
N interactions after sorting for unique pairs: 915312
N genes in interaction map: 19004


,A1_entrez,A2_entrez
0,6204,11224
1,10607,11103
2,54536,60412
3,6147,6159
4,54536,55763
...,...,...
1830039,3439,57864
1830050,27161,266743
1830175,3646,79752
1830325,7979,57804


In [21]:
## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df

In [22]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

biomarker_query = main_file_csv[['entrez_id_biomarker','entrez_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'entrez_id_biomarker': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
biomarker_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,biomarker_query)
biomarker_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_biomarker_query_string.csv',index=False)
biomarker_query_fet_shared_string_ppi

N. gene pairs that interact: 6007
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5728.0,5296.0,True,"{55300, 4615, 8202, 3082, 2064, 2065, 2066, 71...",1298,178,0.137134,114.152228
1,5728.0,3636.0,True,"{200576, 26499, 3716, 3717, 3718, 387, 4233, 5...",1152,57,0.049479,34.068956
2,5728.0,7249.0,True,"{4609, 1026, 1027, 1029, 1019, 2064, 2065, 309...",1202,181,0.150582,144.356054
3,5728.0,4771.0,True,"{6657, 4609, 1029, 1030, 3082, 3084, 2064, 206...",1158,95,0.082038,68.669701
4,5728.0,63932.0,False,"{1457, 283106, 1459}",1142,3,0.002627,-0.097172
...,...,...,...,...,...,...,...,...
144075,4893.0,440590.0,False,{},621,0,0.000000,0.000000
144076,4893.0,79699.0,False,{},626,0,0.000000,-0.202768
144077,4893.0,7791.0,False,"{4609, 391, 3725, 409, 7837, 2335, 1956, 1969,...",699,21,0.030043,9.351460
144078,4893.0,23140.0,False,{},624,0,0.000000,-0.203459


In [23]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target1_query = main_file_csv[['entrez_id_target1','entrez_id_query']]
target1_query = target1_query.rename(columns={
    'entrez_id_target1': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target1_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,target1_query)
target1_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target1_query_string.csv',index=False)
target1_query_fet_shared_string_ppi

N. gene pairs that interact: 4529
N paralog pairs w/ 1+ interactor (A1 and/or A2): 143870
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5291.0,5296.0,True,"{118788, 51206, 8202, 2064, 2065, 2066, 7189, ...",470,281,0.597872,323.306215
1,5291.0,3636.0,True,"{146433, 26499, 3716, 3717, 3718, 387, 4233, 7...",415,69,0.166265,83.017283
2,5291.0,7249.0,True,"{30849, 1026, 387, 2308, 1029, 2309, 5894, 384...",583,75,0.128645,62.190596
3,5291.0,4771.0,False,"{387, 1029, 3845, 4233, 2064, 2065, 2066, 5781...",487,43,0.088296,34.650616
4,5291.0,63932.0,False,{},420,0,0.000000,-0.188936
...,...,...,...,...,...,...,...,...
144075,5604.0,440590.0,False,{},414,0,0.000000,0.000000
144076,5604.0,79699.0,False,{},419,0,0.000000,0.000000
144077,5604.0,7791.0,False,"{4609, 387, 3725, 6416, 409, 7837, 2335, 28964...",489,24,0.049080,16.005291
144078,5604.0,23140.0,False,{},417,0,0.000000,0.000000


In [24]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target2_query = main_file_csv[['entrez_id_target2','entrez_id_query']]
target2_query = target2_query.rename(columns={
    'entrez_id_target2': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target2_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,target2_query)
target2_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target2_query_string.csv',index=False)
target2_query_fet_shared_string_ppi

N. gene pairs that interact: 932
N paralog pairs w/ 1+ interactor (A1 and/or A2): 143396
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,NaN,5296.0,False,{},407,0,0.000000,0.000000
1,NaN,3636.0,False,{},140,0,0.000000,0.000000
2,NaN,7249.0,False,{},314,0,0.000000,0.000000
3,NaN,4771.0,False,{},184,0,0.000000,0.000000
4,NaN,63932.0,False,{},74,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...
144075,5605.0,440590.0,False,{},213,0,0.000000,0.000000
144076,5605.0,79699.0,False,{},218,0,0.000000,0.000000
144077,5605.0,7791.0,False,"{387, 1956, 998, 3690, 3725, 207, 54518, 7414,...",298,14,0.046980,10.589875
144078,5605.0,23140.0,False,{},216,0,0.000000,0.000000
